# Training Pipeline Walkthrough: Social Media Disaster Dataset

This notebook shows a simple step-by-step flow using:
`ai-ml/datasets/socialmedia-disaster-tweets-DFE.csv`

Goal:
1. Read the new dataset
2. Prepare a clean training file
3. Build config
4. Run pipeline
5. Compare two models


## Step 0: Setup

In [1]:
from pathlib import Path
import json
import sys

print("[Step 0] Finding project paths...")
TRAINING_ROOT = Path.cwd().resolve()
if not (TRAINING_ROOT / "src").exists():
    for p in [TRAINING_ROOT, *TRAINING_ROOT.parents]:
        candidate = p / "ai-ml" / "training_pipeline"
        if (candidate / "src").exists():
            TRAINING_ROOT = candidate
            break

REPO_ROOT = TRAINING_ROOT.parent.parent

def repo_rel(path: Path) -> str:
    try:
        return str(path.resolve().relative_to(REPO_ROOT.resolve()))
    except Exception:
        return str(path)

DATASET_REL_PATH = Path("..") / "datasets" / "socialmedia-disaster-tweets-DFE.csv"
DATASET_PATH = (TRAINING_ROOT / DATASET_REL_PATH).resolve()

if str(TRAINING_ROOT) not in sys.path:
    sys.path.insert(0, str(TRAINING_ROOT))

print("TRAINING_ROOT:", repo_rel(TRAINING_ROOT))
print("DATASET_REL_PATH (from training_pipeline):", DATASET_REL_PATH.as_posix())
print("DATASET_PATH (from repo root):", repo_rel(DATASET_PATH))
print("DATASET_PATH exists:", DATASET_PATH.exists())


[Step 0] Finding project paths...
TRAINING_ROOT: ai-ml\training_pipeline
DATASET_REL_PATH (from training_pipeline): ../datasets/socialmedia-disaster-tweets-DFE.csv
DATASET_PATH (from repo root): ai-ml\datasets\socialmedia-disaster-tweets-DFE.csv
DATASET_PATH exists: True


## Step 1: Inspect the raw dataset
Use `latin1` encoding (this file is not UTF-8).

In [2]:
import pandas as pd

print("[Step 1] Loading raw dataset...")
raw_df = pd.read_csv(DATASET_PATH, encoding="latin1")

print("Shape:", raw_df.shape)
print("Columns:")
for c in raw_df.columns:
    print(" -", c)

print("\nTarget distribution (choose_one):")
print(raw_df["choose_one"].value_counts(dropna=False).head(10))

print("\nSample rows:")
display(raw_df[["choose_one", "choose_one:confidence", "text", "keyword", "location"]].head(5))


[Step 1] Loading raw dataset...
Shape: (10876, 13)
Columns:
 - _unit_id
 - _golden
 - _unit_state
 - _trusted_judgments
 - _last_judgment_at
 - choose_one
 - choose_one:confidence
 - choose_one_gold
 - keyword
 - location
 - text
 - tweetid
 - userid

Target distribution (choose_one):
choose_one
Not Relevant    6187
Relevant        4673
Can't Decide      16
Name: count, dtype: int64

Sample rows:


,choose_one,choose_one:confidence,text,keyword,location
0,Relevant,1.0000,Just happened a terrible car crash,NaN,NaN
1,Relevant,1.0000,Our Deeds are the Reason of this #earthquake M...,NaN,NaN
2,Relevant,1.0000,"Heard about #earthquake is different cities, s...",NaN,NaN
3,Relevant,0.9603,"there is a forest fire at spot pond, geese are...",NaN,NaN
4,Relevant,1.0000,Forest fire near La Ronge Sask. Canada,NaN,NaN


## Step 2: Prepare a simple training dataset
We create a binary target and a few numeric features to keep training fast and easy to follow.

In [3]:
print("[Step 2] Building prepared dataset...")

df = raw_df.copy()
df["label"] = (df["choose_one"].astype(str).str.strip().str.lower() == "relevant").astype(int)
df["text_len"] = df["text"].fillna("").astype(str).str.len()
df["keyword_len"] = df["keyword"].fillna("").astype(str).str.len()
df["has_location"] = df["location"].notna().astype(int)
df["judge_confidence"] = pd.to_numeric(df["choose_one:confidence"], errors="coerce").fillna(0.0)
df["trusted_judgments"] = pd.to_numeric(df["_trusted_judgments"], errors="coerce").fillna(0.0)

prepared_cols = [
    "text_len",
    "keyword_len",
    "has_location",
    "judge_confidence",
    "trusted_judgments",
    "label",
]
prepared_df = df[prepared_cols].copy()

artifacts_dir = TRAINING_ROOT / "artifacts"
artifacts_dir.mkdir(parents=True, exist_ok=True)
prepared_dataset_rel_path = Path("artifacts") / "socialmedia_disaster_prepared.csv"
prepared_dataset_path = TRAINING_ROOT / prepared_dataset_rel_path
prepared_df.to_csv(prepared_dataset_path, index=False)

print("Prepared dataset saved (from repo root):", repo_rel(prepared_dataset_path))
print("Prepared shape:", prepared_df.shape)
print("Label distribution:")
print(prepared_df["label"].value_counts())
display(prepared_df.head(5))


[Step 2] Building prepared dataset...
Prepared dataset saved (from repo root): ai-ml\training_pipeline\artifacts\socialmedia_disaster_prepared.csv
Prepared shape: (10876, 6)
Label distribution:
label
0    6203
1    4673
Name: count, dtype: int64


,text_len,keyword_len,has_location,judge_confidence,trusted_judgments,label
0,34,0,0,1.0000,156,1
1,69,0,0,1.0000,152,1
2,64,0,0,1.0000,137,1
3,96,0,0,0.9603,136,1
4,38,0,0,1.0000,138,1


## Step 3: Create config for this dataset
This config uses the prepared CSV and trains a Random Forest first.

In [4]:
print("[Step 3] Creating run config...")

run_config = {
    "dataset": {
        "path": prepared_dataset_rel_path.as_posix(),
        "abnormal_path": "",
        "target_column": "label",
        "train_split": 0.7,
        "val_split": 0.15,
        "test_split": 0.15,
        "random_seed": 42,
        "stratify": True,
    },
    "preprocessing": {
        "missing_value_strategy": "mean",
        "normalization": "standard",
        "encoding": "none",
        "feature_selection": False,
    },
    "model": {
        "type": "random_forest",
        "hyperparameters": {"n_estimators": 120, "max_depth": 8},
    },
    "training": {
        "batch_size": 32,
        "epochs": 10,
        "learning_rate": 0.001,
        "verbose": True,
    },
    "output": {
        "path": "checkpoints",
        "log_path": "logs",
        "save_best_only": True,
        "checkpoint_prefix": "socialmedia",
    },
}

config_rel_path = Path("configs") / "socialmedia_disaster_config.json"
config_path = TRAINING_ROOT / config_rel_path
config_path.write_text(json.dumps(run_config, indent=2), encoding="utf-8")

print("Config saved (from repo root):", repo_rel(config_path))
print(json.dumps(run_config["model"], indent=2))


[Step 3] Creating run config...
Config saved (from repo root): ai-ml\training_pipeline\configs\socialmedia_disaster_config.json
{
  "type": "random_forest",
  "hyperparameters": {
    "n_estimators": 120,
    "max_depth": 8
  }
}


## Step 4: Run the pipeline end-to-end

In [5]:
from src import run_training_pipeline

print("[Step 4] Running training pipeline...")
rf_result = run_training_pipeline(
    config_path=config_path,
    preprocessing_config_path=None,
    run_id="socialmedia_rf_notebook",
    save_checkpoint=False,
)

print("Run ID:", rf_result["run_id"])
print("Rows:", rf_result["rows"])
print("Validation metrics:")
print(json.dumps(rf_result["metrics"]["validation"], indent=2, default=str))
print("Test metrics:")
print(json.dumps(rf_result["metrics"]["test"], indent=2, default=str))


[Step 4] Running training pipeline...


2026-04-21 23:41:15,184 | INFO | training_pipeline | CONFIG SNAPSHOT
2026-04-21 23:41:15,185 | INFO | training_pipeline | CONFIG | section=dataset | value={"path": "artifacts/socialmedia_disaster_prepared.csv", "abnormal_path": "", "target_column": "label", "train_split": 0.7, "val_split": 0.15, "test_split": 0.15, "random_seed": 42, "stratify": true}
2026-04-21 23:41:15,186 | INFO | training_pipeline | CONFIG | section=preprocessing | value={"missing_value_strategy": "mean", "normalization": "standard", "encoding": "none", "feature_selection": false, "selected_features": []}
2026-04-21 23:41:15,186 | INFO | training_pipeline | CONFIG | section=model | value={"type": "random_forest", "hyperparameters": {"n_estimators": 120, "max_depth": 8}, "name": "", "task_type": ""}
2026-04-21 23:41:15,187 | INFO | training_pipeline | CONFIG | section=training | value={"batch_size": 32, "epochs": 10, "learning_rate": 0.001, "verbose": true, "tensorboard_enabled": false, "tensorboard_log_dir": "logs/

Finished training sklearn model: random_forest
Run ID: socialmedia_rf_notebook
Rows: {'input': 10876, 'processed': 10876, 'train': 7613, 'val': 1631, 'test': 1632}
Validation metrics:
{
  "accuracy": 0.688534641324341,
  "precision": 0.7168539325842697,
  "recall": 0.4550641940085592,
  "f1": 0.5567190226876091,
  "confusion_matrix": [
    [
      804,
      126
    ],
    [
      382,
      319
    ]
  ],
  "auc": 0.7511811084012088
}
Test metrics:
{
  "accuracy": 0.6678921568627451,
  "precision": 0.6673684210526316,
  "recall": 0.4522111269614836,
  "f1": 0.5391156462585034,
  "confusion_matrix": [
    [
      773,
      158
    ],
    [
      384,
      317
    ]
  ],
  "auc": 0.7322468592512461
}


## Step 5: Try a PyTorch classification model (Simple MLP)
Reuse same data + config, then switch to a PyTorch model.


In [6]:
import copy

print("[Step 5] Running PyTorch MLP for comparison...")

feature_count = prepared_df.drop(columns=["label"]).shape[1]
print("Detected input_dim for MLP:", feature_count)

torch_config = copy.deepcopy(run_config)
torch_config["model"] = {
    "type": "pytorch_mlp",
    "task_type": "classification",
    "hyperparameters": {
        "input_dim": int(feature_count),
        "hidden_dim": 64,
        "output_dim": 2,
    },
}
torch_config["training"]["epochs"] = 30

torch_config_rel_path = Path("configs") / "socialmedia_disaster_config_pytorch.json"
torch_config_path = TRAINING_ROOT / torch_config_rel_path
torch_config_path.write_text(json.dumps(torch_config, indent=2), encoding="utf-8")
print("PyTorch config saved (from repo root):", repo_rel(torch_config_path))

try:
    torch_result = run_training_pipeline(
        config_path=torch_config_path,
        preprocessing_config_path=None,
        run_id="socialmedia_pytorch_notebook",
        save_checkpoint=False,
    )
    print("Random Forest test accuracy:", rf_result["metrics"]["test"].get("accuracy"))
    print("PyTorch MLP test accuracy:", torch_result["metrics"]["test"].get("accuracy"))
except ImportError as exc:
    print("PyTorch run skipped (dependency issue):", exc)


2026-04-21 23:41:15,914 | INFO | training_pipeline | CONFIG SNAPSHOT
2026-04-21 23:41:15,915 | INFO | training_pipeline | CONFIG | section=dataset | value={"path": "artifacts/socialmedia_disaster_prepared.csv", "abnormal_path": "", "target_column": "label", "train_split": 0.7, "val_split": 0.15, "test_split": 0.15, "random_seed": 42, "stratify": true}
2026-04-21 23:41:15,916 | INFO | training_pipeline | CONFIG | section=preprocessing | value={"missing_value_strategy": "mean", "normalization": "standard", "encoding": "none", "feature_selection": false, "selected_features": []}
2026-04-21 23:41:15,917 | INFO | training_pipeline | CONFIG | section=model | value={"type": "pytorch_mlp", "task_type": "classification", "hyperparameters": {"input_dim": 5, "hidden_dim": 64, "output_dim": 2}, "name": ""}
2026-04-21 23:41:15,918 | INFO | training_pipeline | CONFIG | section=training | value={"batch_size": 32, "epochs": 30, "learning_rate": 0.001, "verbose": true, "tensorboard_enabled": false, "te

[Step 5] Running PyTorch MLP for comparison...
Detected input_dim for MLP: 5
PyTorch config saved (from repo root): ai-ml\training_pipeline\configs\socialmedia_disaster_config_pytorch.json
Epoch 1: loss=0.6506
Epoch 2: loss=0.6422
Epoch 3: loss=0.6402
Epoch 4: loss=0.6384
Epoch 5: loss=0.6379
Epoch 6: loss=0.6357
Epoch 7: loss=0.6347
Epoch 8: loss=0.6349
Epoch 9: loss=0.6345
Epoch 10: loss=0.6330
Epoch 11: loss=0.6326
Epoch 12: loss=0.6316
Epoch 13: loss=0.6317
Epoch 14: loss=0.6309
Epoch 15: loss=0.6302
Epoch 16: loss=0.6294
Epoch 17: loss=0.6288
Epoch 18: loss=0.6284
Epoch 19: loss=0.6284
Epoch 20: loss=0.6282
Epoch 21: loss=0.6279
Epoch 22: loss=0.6279
Epoch 23: loss=0.6259
Epoch 24: loss=0.6264
Epoch 25: loss=0.6256
Epoch 26: loss=0.6257
Epoch 27: loss=0.6248
Epoch 28: loss=0.6248
Epoch 29: loss=0.6246


2026-04-21 23:41:26,497 | INFO | training_pipeline | Training complete | model_name=simple_mlp | model_type=pytorch | task_type=classification
2026-04-21 23:41:26,517 | INFO | training_pipeline | METRIC | metric=val_accuracy | value=0.6204782342121398
2026-04-21 23:41:26,519 | INFO | training_pipeline | METRIC | metric=val_precision | value=0.5813492063492064
2026-04-21 23:41:26,519 | INFO | training_pipeline | METRIC | metric=val_recall | value=0.41797432239657634
2026-04-21 23:41:26,520 | INFO | training_pipeline | METRIC | metric=val_f1 | value=0.4863070539419087
2026-04-21 23:41:26,521 | INFO | training_pipeline | METRIC | metric=val_confusion_matrix | value=[[719, 211], [408, 293]]
2026-04-21 23:41:26,522 | INFO | training_pipeline | METRIC | metric=val_auc | value=0.6681998067277162
2026-04-21 23:41:26,522 | INFO | training_pipeline | METRIC | metric=test_accuracy | value=0.6078431372549019
2026-04-21 23:41:26,523 | INFO | training_pipeline | METRIC | metric=test_precision | valu

Epoch 30: loss=0.6242
Random Forest test accuracy: 0.6678921568627451
PyTorch MLP test accuracy: 0.6078431372549019


## Step 5B: Run PyTorch with TensorBoard (alongside training)

This starts TensorBoard (if available) and then runs a TensorBoard-enabled PyTorch pipeline run.


In [7]:
from IPython import get_ipython

print('[Step 5B] Running PyTorch + TensorBoard...')

torch_tb_config = copy.deepcopy(torch_config)
torch_tb_config['training']['tensorboard_enabled'] = True
torch_tb_config['training']['tensorboard_log_dir'] = 'logs/tensorboard'
torch_config["training"]["epochs"] = 200

torch_tb_config_rel_path = Path('configs') / 'socialmedia_disaster_config_pytorch_tensorboard_notebook.json'
torch_tb_config_path = TRAINING_ROOT / torch_tb_config_rel_path
torch_tb_config_path.write_text(json.dumps(torch_tb_config, indent=2), encoding='utf-8')
print('PyTorch+TensorBoard config saved (from repo root):', repo_rel(torch_tb_config_path))

logdir = (TRAINING_ROOT / 'logs' / 'tensorboard').resolve()
logdir.mkdir(parents=True, exist_ok=True)

torch_tb_result = run_training_pipeline(
    config_path=torch_tb_config_path,
    preprocessing_config_path=None,
    run_id='socialmedia_pytorch_tb_notebook',
    save_checkpoint=False,
)

print('TensorBoard status:', torch_tb_result['training'].get('tensorboard_status'))
print('TensorBoard log dir:', torch_tb_result['training'].get('tensorboard_log_dir'))
print('PyTorch+TB test accuracy:', torch_tb_result['metrics']['test'].get('accuracy'))

# Use Jupyter TensorBoard extension in-notebook.
ip = get_ipython()
if ip is not None:
    ip.run_line_magic('load_ext', 'tensorboard')
    ip.run_line_magic('tensorboard', f'--logdir "{logdir}" --port 6006')
else:
    print('Not running inside IPython/Jupyter; start TensorBoard manually.')



2026-04-21 23:41:26,567 | INFO | training_pipeline | CONFIG SNAPSHOT
2026-04-21 23:41:26,568 | INFO | training_pipeline | CONFIG | section=dataset | value={"path": "artifacts/socialmedia_disaster_prepared.csv", "abnormal_path": "", "target_column": "label", "train_split": 0.7, "val_split": 0.15, "test_split": 0.15, "random_seed": 42, "stratify": true}
2026-04-21 23:41:26,569 | INFO | training_pipeline | CONFIG | section=preprocessing | value={"missing_value_strategy": "mean", "normalization": "standard", "encoding": "none", "feature_selection": false, "selected_features": []}
2026-04-21 23:41:26,570 | INFO | training_pipeline | CONFIG | section=model | value={"type": "pytorch_mlp", "task_type": "classification", "hyperparameters": {"input_dim": 5, "hidden_dim": 64, "output_dim": 2}, "name": ""}
2026-04-21 23:41:26,571 | INFO | training_pipeline | CONFIG | section=training | value={"batch_size": 32, "epochs": 30, "learning_rate": 0.001, "verbose": true, "tensorboard_enabled": true, "ten

2026-04-21 23:41:26,581 | INFO | training_pipeline | Dataset loaded | main_rows=10876 | abnormal_rows=0 | combined_rows=10876
2026-04-21 23:41:26,587 | INFO | training_pipeline | Preprocessing complete | processed_rows=10876 | processed_columns=6 | preprocessing_events=7
2026-04-21 23:41:26,597 | INFO | training_pipeline | Training started | model_name=simple_mlp | model_type=pytorch | epochs=30 | batch_size=32 | train_rows=7613


[Step 5B] Running PyTorch + TensorBoard...
PyTorch+TensorBoard config saved (from repo root): ai-ml\training_pipeline\configs\socialmedia_disaster_config_pytorch_tensorboard_notebook.json
Epoch 1: loss=0.6506
Epoch 2: loss=0.6422
Epoch 3: loss=0.6402
Epoch 4: loss=0.6384
Epoch 5: loss=0.6379
Epoch 6: loss=0.6357
Epoch 7: loss=0.6347
Epoch 8: loss=0.6349
Epoch 9: loss=0.6345
Epoch 10: loss=0.6330
Epoch 11: loss=0.6326
Epoch 12: loss=0.6316
Epoch 13: loss=0.6317
Epoch 14: loss=0.6309
Epoch 15: loss=0.6302
Epoch 16: loss=0.6294
Epoch 17: loss=0.6288
Epoch 18: loss=0.6284
Epoch 19: loss=0.6284
Epoch 20: loss=0.6282
Epoch 21: loss=0.6279
Epoch 22: loss=0.6279
Epoch 23: loss=0.6259
Epoch 24: loss=0.6264
Epoch 25: loss=0.6256
Epoch 26: loss=0.6257
Epoch 27: loss=0.6248
Epoch 28: loss=0.6248
Epoch 29: loss=0.6246


2026-04-21 23:41:35,457 | INFO | training_pipeline | Training complete | model_name=simple_mlp | model_type=pytorch | task_type=classification
2026-04-21 23:41:35,478 | INFO | training_pipeline | METRIC | metric=val_accuracy | value=0.6204782342121398
2026-04-21 23:41:35,479 | INFO | training_pipeline | METRIC | metric=val_precision | value=0.5813492063492064
2026-04-21 23:41:35,480 | INFO | training_pipeline | METRIC | metric=val_recall | value=0.41797432239657634
2026-04-21 23:41:35,481 | INFO | training_pipeline | METRIC | metric=val_f1 | value=0.4863070539419087
2026-04-21 23:41:35,482 | INFO | training_pipeline | METRIC | metric=val_confusion_matrix | value=[[719, 211], [408, 293]]
2026-04-21 23:41:35,483 | INFO | training_pipeline | METRIC | metric=val_auc | value=0.6681998067277162
2026-04-21 23:41:35,484 | INFO | training_pipeline | METRIC | metric=test_accuracy | value=0.6078431372549019
2026-04-21 23:41:35,485 | INFO | training_pipeline | METRIC | metric=test_precision | valu

Epoch 30: loss=0.6242
TensorBoard status: enabled
TensorBoard log dir: C:\Users\koolg\OneDrive - Deakin University\PHOENIX\Phoenix Github Repo\ai-ml\training_pipeline\logs\tensorboard\socialmedia_pytorch_tb_notebook
PyTorch+TB test accuracy: 0.6078431372549019


Reusing TensorBoard on port 6006 (pid 18556), started 0:42:23 ago. (Use '!kill 18556' to kill it.)

## Step 6: Check logs and artifacts
These files help others reproduce the same run.

In [8]:
print("[Step 6] Listing generated files...")
print("Artifacts:")
for p in sorted(artifacts_dir.glob("socialmedia_disaster_*")):
    print(" -", repo_rel(p))

print("\nConfigs:")
for p in sorted((TRAINING_ROOT / "configs").glob("*socialmedia*.json")):
    print(" -", repo_rel(p))

print("\nLatest logs:")
for p in sorted((TRAINING_ROOT / "logs").glob("*.log"))[-5:]:
    print(" -", repo_rel(p))


[Step 6] Listing generated files...
Artifacts:
 - ai-ml\training_pipeline\artifacts\socialmedia_disaster_prepared.csv

Configs:
 - ai-ml\training_pipeline\configs\socialmedia_disaster_config.json
 - ai-ml\training_pipeline\configs\socialmedia_disaster_config_lr.json
 - ai-ml\training_pipeline\configs\socialmedia_disaster_config_pytorch.json
 - ai-ml\training_pipeline\configs\socialmedia_disaster_config_pytorch_tensorboard_notebook.json
 - ai-ml\training_pipeline\configs\socialmedia_disaster_pytorch_tensorboard.json

Latest logs:
 - ai-ml\training_pipeline\logs\local_pytorch_tb_run.log
 - ai-ml\training_pipeline\logs\socialmedia_pytorch_notebook.log
 - ai-ml\training_pipeline\logs\socialmedia_pytorch_smoke.log
 - ai-ml\training_pipeline\logs\socialmedia_pytorch_tb_notebook.log
 - ai-ml\training_pipeline\logs\socialmedia_rf_notebook.log


## How Others Can Reuse This

- Put a CSV in `ai-ml/datasets/`
- Update dataset path and target logic in **Step 2**
- Keep the same workflow (prepare -> config -> run -> compare)
- Re-run all cells from top to bottom
